<a href="https://colab.research.google.com/github/Anupampal1992/Group-Project/blob/EDA/Traffic_Accident.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [299]:
!git clone  https://github.com/Anupampal1992/Group-Project.git
%cd Group-Project
!ls

Cloning into 'Group-Project'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 30 (delta 17), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 188.41 KiB | 3.09 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project
 Traffic_Accident.ipynb  'Traffic Accident Severity Predictor Dataset.csv'


In [300]:

!git status


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [301]:
import pandas as pd
import numpy as np

#visualization
import matplotlib.pyplot as plt
import seaborn as sns

#preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from scipy.spatial import transform


#Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

#Evaluation
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score,accuracy_score

#Pipeline
from sklearn.pipeline import Pipeline

In [302]:
df=pd.read_csv('Traffic Accident Severity Predictor Dataset.csv')  #Loading the data from Github
df.head(5)

,Weather,Road_Type,Time_of_Day,Traffic_Density,Speed_Limit,Number_of_Vehicles,Driver_Alcohol,Accident_Severity,Road_Condition,Vehicle_Type,Driver_Age,Driver_Experience,Road_Light_Condition,Accident
0,Rainy,City Road,Morning,1.0,100.0,5.0,0.0,NaN,Wet,Car,51.0,48.0,Artificial Light,0
1,Clear,Rural Road,Night,NaN,120.0,3.0,0.0,Moderate,Wet,Truck,49.0,43.0,Artificial Light,0
2,Rainy,Highway,Evening,1.0,60.0,4.0,0.0,Low,Icy,Car,54.0,52.0,Artificial Light,0
3,Clear,City Road,Afternoon,2.0,60.0,3.0,0.0,Low,Under Construction,Bus,34.0,31.0,Daylight,0
4,Rainy,Highway,Morning,1.0,195.0,11.0,0.0,Low,Dry,Car,62.0,55.0,Artificial Light,1


EDA

In [303]:
df.shape

(20000, 14)

In [304]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Weather               19958 non-null  object 
 1   Road_Type             19958 non-null  object 
 2   Time_of_Day           19958 non-null  object 
 3   Traffic_Density       19958 non-null  float64
 4   Speed_Limit           19958 non-null  float64
 5   Number_of_Vehicles    19958 non-null  float64
 6   Driver_Alcohol        19958 non-null  float64
 7   Accident_Severity     19958 non-null  object 
 8   Road_Condition        19958 non-null  object 
 9   Vehicle_Type          19958 non-null  object 
 10  Driver_Age            19958 non-null  float64
 11  Driver_Experience     19958 non-null  float64
 12  Road_Light_Condition  19958 non-null  object 
 13  Accident              20000 non-null  int64  
dtypes: float64(6), int64(1), object(7)
memory usage: 2.1+ MB


In [305]:
df.describe(include="all")

,Weather,Road_Type,Time_of_Day,Traffic_Density,Speed_Limit,Number_of_Vehicles,Driver_Alcohol,Accident_Severity,Road_Condition,Vehicle_Type,Driver_Age,Driver_Experience,Road_Light_Condition,Accident
count,19958,19958,19958,19958.000000,19958.000000,19958.000000,19958.000000,19958,19958,19958,19958.000000,19958.000000,19958,20000.000000
unique,5,4,4,NaN,NaN,NaN,NaN,3,4,4,NaN,NaN,3,NaN
top,Clear,Highway,Afternoon,NaN,NaN,NaN,NaN,Low,Dry,Car,NaN,NaN,Artificial Light,NaN
freq,8366,10116,6826,NaN,NaN,NaN,NaN,11862,10072,14805,NaN,NaN,9978,NaN
mean,NaN,NaN,NaN,1.010923,71.448943,3.282694,0.164445,NaN,NaN,NaN,43.146758,38.859154,NaN,0.292000
std,NaN,NaN,NaN,0.783966,32.366260,1.999111,0.370688,NaN,NaN,NaN,15.099349,15.249536,NaN,0.454694
min,NaN,NaN,NaN,0.000000,30.000000,1.000000,0.000000,NaN,NaN,NaN,18.000000,9.000000,NaN,0.000000
25%,NaN,NaN,NaN,0.000000,50.000000,2.000000,0.000000,NaN,NaN,NaN,30.000000,26.000000,NaN,0.000000
50%,NaN,NaN,NaN,1.000000,60.000000,3.000000,0.000000,NaN,NaN,NaN,43.000000,39.000000,NaN,0.000000
75%,NaN,NaN,NaN,2.000000,80.000000,4.000000,0.000000,NaN,NaN,NaN,56.000000,52.000000,NaN,1.000000


In [306]:
df['Accident_Severity'].value_counts()  #Target variable

,count
Accident_Severity,
Low,11862
Moderate,6090
High,2006


In [307]:
df.isnull().sum()   #cheacking the null value

,0
Weather,42
Road_Type,42
Time_of_Day,42
Traffic_Density,42
Speed_Limit,42
Number_of_Vehicles,42
Driver_Alcohol,42
Accident_Severity,42
Road_Condition,42
Vehicle_Type,42


In [308]:
#Separating the numerical and categorical columns
num_col=df.select_dtypes(include=['int64','float64']).columns
cat_col=df.select_dtypes(include=['object']).columns

In [309]:
# Fillig the missing values
df[num_col] = df[num_col].fillna(df[num_col].median())
df[cat_col] = df[cat_col].fillna(df[cat_col].mode().iloc[0])
print("Numerical Coulumns", df[num_col].columns)
print("Categorical Columns", df[cat_col].columns)
df.isnull().sum()

Numerical Coulumns Index(['Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles',
       'Driver_Alcohol', 'Driver_Age', 'Driver_Experience', 'Accident'],
      dtype='object')
Categorical Columns Index(['Weather', 'Road_Type', 'Time_of_Day', 'Accident_Severity',
       'Road_Condition', 'Vehicle_Type', 'Road_Light_Condition'],
      dtype='object')


,0
Weather,0
Road_Type,0
Time_of_Day,0
Traffic_Density,0
Speed_Limit,0
Number_of_Vehicles,0
Driver_Alcohol,0
Accident_Severity,0
Road_Condition,0
Vehicle_Type,0


**Feature and target separation**

In [310]:
x=df.drop(columns=['Accident_Severity'], axis=1)
y=df['Accident_Severity']
print(x.shape)
print(y.shape)
x.columns


(20000, 13)
(20000,)


Index(['Weather', 'Road_Type', 'Time_of_Day', 'Traffic_Density', 'Speed_Limit',
       'Number_of_Vehicles', 'Driver_Alcohol', 'Road_Condition',
       'Vehicle_Type', 'Driver_Age', 'Driver_Experience',
       'Road_Light_Condition', 'Accident'],
      dtype='object')

In [311]:
y.count()

np.int64(20000)

In [312]:

print("Clases of Target Variable:", y.value_counts())


Clases of Target Variable: Accident_Severity
Low         11904
Moderate     6090
High         2006
Name: count, dtype: int64


In [313]:
order = {'Low': 0, 'Moderate': 1, 'High': 2}
y = y.map(order)
y

,Accident_Severity
0,0
1,1
2,0
3,0
4,0
...,...
19995,1
19996,1
19997,1
19998,0


In [314]:
#get the numerical features and categorical features without Target variable
numerical_features = x.select_dtypes(include=['int64', 'float64']).columns
categorical_features = x.select_dtypes(include=['object']).columns
print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

Numerical Features: Index(['Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles',
       'Driver_Alcohol', 'Driver_Age', 'Driver_Experience', 'Accident'],
      dtype='object')
Categorical Features: Index(['Weather', 'Road_Type', 'Time_of_Day', 'Road_Condition', 'Vehicle_Type',
       'Road_Light_Condition'],
      dtype='object')


In [315]:
# Apply different processing step to different columns group
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

**Split and Train the data**

In [316]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42,  stratify=y
)

print("Shape of x_train:", x_train.shape)
print("Shape of x_test:", x_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)
y_train.value_counts()

Shape of x_train: (16000, 13)
Shape of x_test: (4000, 13)
Shape of y_train: (16000,)
Shape of y_test: (4000,)


,count
Accident_Severity,
0,9523
1,4872
2,1605


**Applying Models**

In [317]:
#Logistic regression
lr = Pipeline(steps=[('preprocessor', preprocessor),
                     ('classifier', LogisticRegression(max_iter=1000))])

lr=lr.fit(x_train, y_train)
y_pred = lr.predict(x_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("accuracy:\n", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[2381    0    0]
 [1218    0    0]
 [ 401    0    0]]
accuracy:
 0.59525
Classification Report:
               precision    recall  f1-score   support

           0       0.60      1.00      0.75      2381
           1       0.00      0.00      0.00      1218
           2       0.00      0.00      0.00       401

    accuracy                           0.60      4000
   macro avg       0.20      0.33      0.25      4000
weighted avg       0.35      0.60      0.44      4000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [318]:
# XGBoost Model
xgb = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', XGBClassifier(
                          objective='multi:softmax',
                         num_class=3,
                          eval_metric='mlogloss',
                          max_depth=3,
                          n_estimators=100,
                          learning_rate=0.1
                      ))])

xgb.fit(x_train, y_train)
y_pred = xgb.predict(x_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("accuracy:\n", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[2376    5    0]
 [1217    1    0]
 [ 401    0    0]]
accuracy:
 0.59425
Classification Report:
               precision    recall  f1-score   support

           0       0.59      1.00      0.75      2381
           1       0.17      0.00      0.00      1218
           2       0.00      0.00      0.00       401

    accuracy                           0.59      4000
   macro avg       0.25      0.33      0.25      4000
weighted avg       0.40      0.59      0.44      4000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [326]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier

# Assuming numerical_features and categorical_features are already defined from previous steps
# (e.g., from cell vLeDCsJc48DG)
# If not, they would need to be defined here again based on x

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        max_depth=6,
        learning_rate=0.05,
        n_estimators=500,
        subsample=0.8,
        colsample_bytree=0.8
    ))
])

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[2258  122    1]
 [1150   66    2]
 [ 369   31    1]]
Accuracy: 0.58125
              precision    recall  f1-score   support

           0       0.60      0.95      0.73      2381
           1       0.30      0.05      0.09      1218
           2       0.25      0.00      0.00       401

    accuracy                           0.58      4000
   macro avg       0.38      0.34      0.28      4000
weighted avg       0.47      0.58      0.46      4000

